In [1]:
# ====== 0.1  mount Google Drive ======
from google.colab import drive
drive.mount('/drive')
%cd /drive/MyDrive/AURA_MoE

# ====== 0.2  install GPU Waymo parser ======
!pip install -q waymo-open-dataset-tf-2-12-0 --no-deps
!pip install -q tensorflow==2.12.0  # align TF versions
#!pip install waymo-open-dataset-tf-2-11-0==1.6.0
#!pip install -q waymo-open-dataset-tf-2-12-0  # matches Colab TF
!pip install -q torch torchvision tensorboard tqdm

# ====== 0.3  standard imports ======
import tensorflow as tf, torch, numpy as np, pathlib, glob, tqdm, pickle, random
from waymo_open_dataset.protos import scenario_pb2
print("✓ imports done, next cell will convert tfrecords → PyTorch tensors")

Mounted at /drive
/drive/MyDrive/AURA_MoE
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 36.1 MB/s eta 0:00:00
ERROR: Could not find a version that satisfies the requirement tensorflow==2.12.0 (from versions: 2.16.0rc0, 2.16.1, 2.16.2, 2.17.0rc0, 2.17.0rc1, 2.17.0, 2.17.1, 2.18.0rc0, 2.18.0rc1, 2.18.0rc2, 2.18.0, 2.18.1, 2.19.0rc0, 2.19.0, 2.19.1, 2.20.0rc0, 2.20.0)
ERROR: No matching distribution found for tensorflow==2.12.0
✓ imports done, next cell will convert tfrecords → PyTorch tensors


In [2]:
# ====== 1.1  expert tagger ======
EXPERT_TAGS = {
    0: ['highway', 'free_flow'],
    1: ['highway', 'cut_in'],
    2: ['urban', 'stop_sign', 'traffic_light'],
    3: ['left_turn', 'no_signal'],
    4: ['night', 'rain']
}
def tag_scenario(sc):
    tags = sc.tags
    for e, kw in EXPERT_TAGS.items():
        if any(k in tags for k in kw):
            return e
    return random.randint(0, 4)   # fallback uniform

In [3]:
# ====== 1.2  tfrecord → tensor writer ======
from pathlib import Path
import torch

DATA_DIR   = Path('data/waymo_tf')
OUTPUT_DIR = Path('data/waymo_pt')
SPLITS     = {'training': 45, 'validation': 10}   # 45 & 10 files you uploaded

# create folders
for split in SPLITS:
    for e in range(5):
        (OUTPUT_DIR / f'E{e}' / split).mkdir(parents=True, exist_ok=True)

def parse_one_record(record):
    sc = scenario_pb2.Scenario()
    sc.ParseFromString(record)
    e     = tag_scenario(sc)
    frames = []
    # ---- ego future 3 s (30 steps @ 10 Hz) ----
    ego_track = next(t for t in sc.tracks if t.id == sc.sdc_id)
    future = np.array([[state.velocity_x, state.velocity_y]
                       for state in ego_track.states][10:40])  # 30 steps
    # ---- dummy image & lidar (replace with real later) ----
    image = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
    lidar = np.random.rand(3, 64, 64).astype(np.float32)
    # ---- ego state (10 dim) ----
    state = np.array([sc.ego_velocity_x, sc.ego_velocity_y,
                      sc.ego_acceleration_x, sc.ego_acceleration_y,
                      sc.ego_heading, 0, 0, 0, 0, 0], dtype=np.float32)
    frames.append({'image': image, 'lidar': lidar, 'state': state, 'future': future})
    return e, frames

In [4]:
# ====== 1.3-debug  ultra-minimal test ======
import tensorflow as tf, pathlib, torch, numpy as np
from waymo_open_dataset.protos import scenario_pb2

DATA_DIR   = pathlib.Path('data/waymo_tf')
tf_file    = sorted(DATA_DIR.glob('training/*.tfrecord'))[0]   # first file only
print('Testing 1 file:', tf_file)

dataset = tf.data.TFRecordDataset(str(tf_file), compression_type='')
count   = 0
for raw in dataset:
    try:
        sc = scenario_pb2.Scenario()
        sc.ParseFromString(raw.numpy())
        print('tracks:', len(sc.tracks), 'sdc_id:', sc.sdc_id)
        if sc.tracks and sc.sdc_id != 0:
            count += 1
        if count == 3:   # stop after 3 good frames
            break
    except Exception as e:
        print('Bad frame – skipping', e)

print('Good frames found:', count)

Testing 1 file: data/waymo_tf/training/1000b16d40810e05.tfrecord
Bad frame – skipping sdc_id
Good frames found: 0


In [5]:
# ====== CARLA overnight generator (balanced 5-expert, 3 M frames) ======
# Runs **headless** on your Windows laptop while you sleep.
# Output:  data/carla_pt/  (same folder tree as Waymo)
# Size:    ≈ 25 GB  (fits your freed space)
# Time:    ≈ 6 h  (3 M frames @ 200 Hz)

!pip install -q carla==0.9.16  # CPU-only, headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.2/34.2 MB 44.9 MB/s eta 0:00:00


In [6]:
# ====== 0.1  config ======
FRAMES_PER_EXPERT = 600_000   # 3 M total
BATCH_SIZE        = 200       # frames per CARLA tick
SAVE_EVERY        = 1_000     # pt files
Town            = 'Town05'   # mixed highway / urban / intersections
WEATHER_CYCLES    = ['ClearNoon', 'ClearNoon', 'WetNoon', 'HardRainNoon', 'ClearNight']  # E0…E4

In [24]:
# ====== 0.2  generator script (save as generate_carla_data.py) ======
SCRIPT = '''
import carla, os, pathlib, torch, random, numpy as np

OUTPUT = pathlib.Path("data/carla_pt")
for e in range(5):
    for split in ["training", "validation"]:
        (OUTPUT / f"E{e}" / split).mkdir(parents=True, exist_ok=True)

client = carla.Client("localhost", 2000)
client.set_timeout(60.0)
world  = client.load_world("Town05")
tm     = client.get_trafficmanager(8000)
weather_list = [
    carla.WeatherParameters.ClearNoon,
    carla.WeatherParameters.ClearNoon,
    carla.WeatherParameters.WetNoon,
    carla.WeatherParameters.HardRainNoon,
    carla.WeatherParameters.ClearNight
]

def get_weather(idx):
    return weather_list[idx % 5]

def tag_scenario(vehicle, world):
    v = vehicle.get_velocity()
    spd = np.hypot(v.x, v.y)
    if spd > 15 and not world.get_traffic_lights():
        return 0
    if abs(v.y) > 2: return 1
    if spd < 8 and world.get_traffic_lights(): return 2
    if abs(vehicle.get_transform().rotation.yaw) > 30: return 3
    return 4

def dummy_sensors(world, vehicle):
    image = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
    lidar = np.random.rand(3, 64, 64).astype(np.float32)
    v = vehicle.get_velocity()
    state = np.array([np.hypot(v.x, v.y), v.x, v.y,
                      vehicle.get_acceleration().x, vehicle.get_acceleration().y,
                      vehicle.get_transform().rotation.yaw, 0, 0, 0, 0], dtype=np.float32)
    future = np.random.rand(30, 2).astype(np.float32)  # dummy 3-s
    return image, lidar, state, future

def main():
    bp = world.get_blueprint_library().filter('vehicle.tesla.model3')[0]
    spawn_points = world.get_map().get_spawn_points()
    weather_idx = 0
    for e in range(5):
        for split in ["training", "validation"]:
            count = 0
            while count < FRAMES_PER_EXPERT:
                world.set_weather(get_weather(e))
                spawn = random.choice(spawn_points)
                vehicle = world.spawn_actor(bp, spawn)
                try:
                    for _ in range(BATCH_SIZE):
                        world.tick()
                        e_tag = tag_scenario(vehicle, world)
                        if e_tag == e:  # keep only frames that match expert
                            img, lidar, state, fut = dummy_sensors(world, vehicle)
                            torch.save({'image': img, 'lidar': lidar, 'state': state, 'future': fut},
                                       OUTPUT / f"E{e}" / split / f"{count:07d}.pt")
                            count += 1
                            if count % SAVE_EVERY == 0:
                                print(f"E{e} {split} {count//1000}k")
                        if count >= FRAMES_PER_EXPERT:
                            break
                finally:
                    vehicle.destroy()
            print(f"✓ E{e} {split} finished – {count} frames")

if __name__ == "__main__":
    main()
'''

# ====== 0.3  write & run ======
with open('generate_carla_data.py', 'w') as f:
    f.write(SCRIPT)

print("Generator saved – start CARLA server then run:")
print("!python generate_carla_data.py")

Generator saved – start CARLA server then run:
!python generate_carla_data.py


In [23]:
# ====== 1.1  launch generator (headless, overnight) ======
# Runs ≈ 6 h → 3 M frames → 25 GB
%cd /drive/MyDrive/AURA_MoE
!python generate_carla_data.py

/drive/MyDrive/AURA_MoE
Traceback (most recent call last):
  File "/drive/MyDrive/AURA_MoE/generate_carla_data.py", line 11, in <module>
    world  = client.load_world("Town05")
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: time-out of 60000ms while waiting for the simulator, make sure the simulator is ready and connected to 417d86324a79.ngrok-free.app:2000
